# Section 3: Hyperparameter Tuning and Validation

In the previous section on the K-Nearest Neighbor algorithm, we identified two critical choices that needed to be made before we could run the classifier: the value of $K$ (number of neighbors) and the choice of the distance metric (L1 vs L2). 

These choices are not learned from the training data via an optimization algorithm; rather, they are set by the researcher ahead of time. These are known as **Hyperparameters**. In this section, we discuss the proper methodology for selecting them.

## 3.1 Defining Hyperparameters

A machine learning algorithm typically has two types of parameters:
1.  **Learnable Parameters:** Weights and biases that the algorithm learns directly from the data during the training process (we will see these in Section 4).
2.  **Hyperparameters:** Settings that control the learning process itself. They must be chosen *before* training begins.

For K-NN, the hyperparameters are:
*   $K$: How many neighbors should vote?
*   **Distance Metric:** L1, L2, or something else (like cosine similarity)?

> [!IMPORTANT]
> Hyperparameters are incredibly problem-dependent and dataset-dependent. There is rarely a single "best" hyperparameter setting that works for all tasks. Therefore, we must *search* for the best settings using our data.

## 3.2 Data Splits: The Golden Rule

How do we figure out which hyperparameters are best? 

### The Bad Idea: Evaluate on Training Data
If we evaluate our hyperparameters on the same data we used to train the model, we will fail catastrophically.
For K-NN, if we evaluate on the training set, $K=1$ will *always* achieve perfect 100% accuracy because the nearest neighbor to any training image is the image itself. However, a $K=1$ model will likely overfit and generalize poorly to unseen data.

### The Slightly Better Idea: Evaluate on Test Data
We could hold out a portion of our data as a "Test Set," train on the rest, and pick the hyperparameters that give the highest accuracy on the Test Set.
**Why is this bad?** It violates the fundamental purpose of the test set. The test set must remain completely unseen until the very end of the project to provide an unbiased estimate of generalization performance. If you use it to tune hyperparameters, you are implicitly "training" on the test set (overfitting to the test set).

### The Correct Idea: Train, Validation, and Test Splits
The golden standard in machine learning is a three-way split:

1.  **Training Set:** Used exclusively to train the algorithm (memorize the data, in K-NN's case).
2.  **Validation Set (or Dev Set):** A fake test set. Used exclusively to evaluate different hyperparameter settings. You train on the Training Set, evaluate on the Validation Set, and pick the hyperparameters that yield the highest validation accuracy.
3.  **Test Set:** Locked away until the very end. Used only once to report the final, unbiased performance of the chosen model.

```mermaid
pie title Data Splitting
    "Training Set (70%)" : 70
    "Validation Set (15%)" : 15
    "Test Set (15%)" : 15
```

In [ ]:
import numpy as np

# Conceptual representation of a dataset
np.random.seed(42)
num_samples = 1000
X_full = np.random.randn(num_samples, 100) # 1000 samples, 100 features
y_full = np.random.randint(0, 10, num_samples) # 10 classes

# 1. Shuffle the data
indices = np.arange(num_samples)
np.random.shuffle(indices)
X_full = X_full[indices]
y_full = y_full[indices]

# 2. Split the data (e.g., 80% train, 10% val, 10% test)
num_train = 800
num_val = 100
num_test = 100

X_train, y_train = X_full[:num_train], y_full[:num_train]
X_val, y_val = X_full[num_train:num_train+num_val], y_full[num_train:num_train+num_val]
X_test, y_test = X_full[-num_test:], y_full[-num_test:]

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}\n")

# Pseudo-code for hyperparameter tuning loop:
best_k = 1
best_val_acc = 0.0

for k in [1, 3, 5, 7, 10]:
    # 1. Train on X_train (For K-NN, this is just memorizing)
    # 2. Predict on X_val using k
    # 3. Calculate validation accuracy
    val_acc = np.random.rand() # Simulated accuracy for illustration
    print(f"Testing K={k} -> Val Accuracy: {val_acc:.2f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_k = k

print(f"\nSelected best hyperparameter: K={best_k}")
# Finally, evaluate ONLY ONCE on the test set (X_test, y_test) using best_k

## 3.3 Cross-Validation

In scenarios where the dataset is extremely small, setting aside a dedicated validation set might leave you with too little data to properly train the model. Furthermore, evaluating on a tiny validation set might be highly sensitive to the specific random split.

To mitigate this, we use **K-Fold Cross-Validation** (Note: The "K" here refers to folds, unrelated to the "K" in K-NN).

### The K-Fold Algorithm
Instead of a single, static validation set:
1.  Set aside the Test Set as usual.
2.  Divide the remaining Training Data into $F$ equal partitions (folds).
3.  Iterate $F$ times. For each iteration $i$:
    *   Use fold $i$ as the Validation Set.
    *   Use the remaining $F-1$ folds combined as the Training Set.
    *   Train the model and record the validation accuracy.
4.  Average the $F$ validation accuracies to get a single, robust performance estimate for that specific hyperparameter setting.

```mermaid
gantt
    title 5-Fold Cross Validation
    dateFormat  X
    axisFormat %s
    section Fold 1
    Validation   :active, 0, 1
    Train        :done, 1, 5
    section Fold 2
    Train        :done, 0, 1
    Validation   :active, 1, 2
    Train        :done, 2, 5
    section Fold 3
    Train        :done, 0, 2
    Validation   :active, 2, 3
    Train        :done, 3, 5
```

> [!NOTE]
> Cross-validation provides a much more statistically reliable estimate of generalization performance. However, because it requires training the model $F$ times for *every* hyperparameter combination, it is computationally expensive. It is heavily used for small datasets and non-parametric algorithms like K-NN, but is generally avoided in deep learning on massive datasets (like ImageNet) due to the extreme computational cost.

---
### Summary of Section 3
*   **Concepts Introduced:** Hyperparameters vs. Learnable Parameters, Train/Validation/Test Splits, Data Leakage (Overfitting to the test set), K-Fold Cross-Validation.
*   **Notation Introduced:** $F$ (Number of folds in cross-validation).
*   **Dependencies for Next Section:** We have concluded the non-parametric paradigm. We will now shift gears entirely into parametric models where parameters are mathematically optimized rather than explicitly searched.